# Chapter 2 — Supervised Learning

Notebook ini mereproduksi dan merangkum konsep utama **Supervised Learning** dari buku *Introduction to Machine Learning with Python*.

## Tujuan Pembelajaran
- Memahami perbedaan **classification** dan **regression**.
- Memahami konsep **generalization**, **overfitting**, dan **underfitting**.
- Menerapkan beberapa algoritma supervised learning menggunakan `scikit-learn`.
- Membandingkan performa model menggunakan data latih dan data uji.

> Catatan: kode dibuat ulang dengan gaya sendiri agar tetap orisinal, tetapi mengikuti alur konsep dari buku.

In [ ]:
# Setup dasar yang digunakan pada sebagian besar chapter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Classification dan Regression

**Classification** digunakan ketika target berupa label/kategori, misalnya benign vs malignant.  
**Regression** digunakan ketika target berupa nilai kontinu, misalnya harga rumah atau nilai prediksi numerik.

Pada notebook ini digunakan:
- `load_breast_cancer()` untuk klasifikasi.
- `load_diabetes()` untuk regresi.

In [ ]:
from sklearn.datasets import load_breast_cancer, load_diabetes

cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target

print("Shape data cancer:", X_cancer.shape)
print("Nama kelas:", cancer.target_names)
print("Jumlah tiap kelas:", np.bincount(y_cancer))

diabetes = load_diabetes()
X_diabetes, y_diabetes = diabetes.data, diabetes.target
print("\nShape data diabetes:", X_diabetes.shape)

## 2. Train-Test Split

Data dipisahkan menjadi data latih dan data uji. Model belajar dari data latih, lalu dievaluasi pada data uji untuk melihat kemampuan generalisasi.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, stratify=y_cancer, random_state=RANDOM_STATE
)
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

## 3. k-Nearest Neighbors Classification

k-NN memprediksi kelas berdasarkan mayoritas kelas dari tetangga terdekat. Nilai `n_neighbors` kecil cenderung kompleks dan rawan overfitting, sedangkan nilai besar membuat model lebih sederhana.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

train_scores = []
test_scores = []
neighbors_range = range(1, 16)

for k in neighbors_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_scores.append(knn.score(X_train, y_train))
    test_scores.append(knn.score(X_test, y_test))

plt.plot(neighbors_range, train_scores, marker='o', label='Akurasi latih')
plt.plot(neighbors_range, test_scores, marker='s', label='Akurasi uji')
plt.xlabel('Jumlah tetangga (k)')
plt.ylabel('Akurasi')
plt.title('Pengaruh k pada KNeighborsClassifier')
plt.legend()
plt.show()

best_k = neighbors_range[int(np.argmax(test_scores))]
print("k terbaik berdasarkan data uji:", best_k)
print("Akurasi uji terbaik:", max(test_scores))

## 4. Logistic Regression

Logistic Regression adalah model linear untuk klasifikasi. Parameter `C` mengontrol regularisasi:
- `C` kecil → regularisasi kuat → model lebih sederhana.
- `C` besar → regularisasi lemah → model lebih kompleks.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

logreg_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
])

logreg_pipe.fit(X_train, y_train)
print("Akurasi latih:", logreg_pipe.score(X_train, y_train))
print("Akurasi uji  :", logreg_pipe.score(X_test, y_test))

## 5. Decision Tree

Decision Tree membuat keputusan melalui struktur pertanyaan `if/else`. Jika pohon terlalu dalam, model bisa menghafal data latih. Oleh karena itu digunakan parameter seperti `max_depth` untuk pre-pruning.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

for depth in [2, 3, 4, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    tree.fit(X_train, y_train)
    print(f"max_depth={depth} | train={tree.score(X_train, y_train):.3f} | test={tree.score(X_test, y_test):.3f}")

### Feature Importance pada Decision Tree

`feature_importances_` menunjukkan fitur mana yang paling banyak berkontribusi dalam pemisahan data pada decision tree.

In [ ]:
tree = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)
tree.fit(X_train, y_train)

importances = pd.Series(tree.feature_importances_, index=cancer.feature_names).sort_values(ascending=False)
print(importances.head(10))

importances.head(10).sort_values().plot(kind='barh')
plt.xlabel('Tingkat kepentingan fitur')
plt.title('10 Feature Importance Tertinggi - Decision Tree')
plt.show()

## 6. Random Forest dan Gradient Boosting

Random Forest menggabungkan banyak decision tree untuk mengurangi overfitting.  
Gradient Boosting membangun tree secara bertahap, di mana tree berikutnya memperbaiki kesalahan tree sebelumnya.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name}")
    print("  Akurasi latih:", round(model.score(X_train, y_train), 3))
    print("  Akurasi uji  :", round(model.score(X_test, y_test), 3))

## 7. Support Vector Machine dan Pentingnya Scaling

SVM sensitif terhadap skala fitur. Karena itu, pipeline dengan `StandardScaler` sangat disarankan.

In [ ]:
from sklearn.svm import SVC

svm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(C=10, gamma='scale', random_state=RANDOM_STATE))
])
svm_pipe.fit(X_train, y_train)
print("Akurasi latih:", svm_pipe.score(X_train, y_train))
print("Akurasi uji  :", svm_pipe.score(X_test, y_test))

## 8. Neural Network / MLP

MLP merupakan jaringan saraf feed-forward sederhana. Model ini membutuhkan scaling dan tuning parameter seperti jumlah hidden layer, jumlah neuron, `alpha`, dan `max_iter`.

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', MLPClassifier(hidden_layer_sizes=(50,), alpha=1.0, max_iter=1000, random_state=RANDOM_STATE))
])
mlp_pipe.fit(X_train, y_train)
print("Akurasi latih:", mlp_pipe.score(X_train, y_train))
print("Akurasi uji  :", mlp_pipe.score(X_test, y_test))

## Kesimpulan Chapter 2

- Supervised learning terdiri dari classification dan regression.
- Evaluasi pada test set penting untuk mengukur generalisasi.
- Model terlalu kompleks dapat overfitting, sedangkan terlalu sederhana dapat underfitting.
- Beberapa model seperti SVM dan neural network sangat membutuhkan scaling.
- Random Forest dan Gradient Boosting sering kuat untuk data tabular.